In [1]:
import numpy as np
import linerate


conductor = linerate.Conductor(
    core_diameter=10.4e-3,
    conductor_diameter=28.1e-3,
    outer_layer_strand_diameter=2.2e-3,
    emissivity=0.9,
    solar_absorptivity=0.9,
    temperature1=25,
    temperature2=75,
    resistance_at_temperature1=7.283e-5,
    resistance_at_temperature2=8.688e-5,
    aluminium_cross_section_area=float("nan"),  # No core magnetisation loss
    constant_magnetic_effect=1,
    current_density_proportional_magnetic_effect=0,
    max_magnetic_core_relative_resistance_increase=1,
)


start_tower = linerate.Tower(latitude=50 - 0.0045, longitude=0, altitude=500 - 88)
end_tower = linerate.Tower(latitude=50 + 0.0045, longitude=0, altitude=500 + 88)
span = linerate.Span(
    conductor=conductor,
    start_tower=start_tower,
    end_tower=end_tower,
    num_conductors=1,
)


weather = linerate.Weather(
    air_temperature=20,
    wind_direction=np.radians(80),  # Conductor azimuth is 0, so angle of attack is 80
    wind_speed=1.66,
    ground_albedo=0.15,
    clearness_ratio=0.5,
)


time_of_measurement = np.datetime64("2016-10-03 14:00")
max_conductor_temperature = 100
current_load = 1000

model = linerate.Cigre601(span, weather, time_of_measurement)
conductor_rating = model.compute_steady_state_ampacity(max_conductor_temperature)
print(f"The span has a steady-state ampacity rating of {conductor_rating:.0f} A if the maximum temperature is {max_conductor_temperature} °C")
conductor_temperature = model.compute_conductor_temperature(current_load)
print(f"The conductor has a temperature of {conductor_temperature:.0f} °C when operated at {current_load} A")

The span has a steady-state ampacity rating of 1505 A if the maximum temperature is 100 °C
The conductor has a temperature of 55 °C when operated at 1000 A


In [1]:
import numpy as np
from datetime import datetime, timedelta

# Longitud de Madrid (grados, Este positivo)
LON_MADRID = -3.7038

def day_of_year(dt):
    """Devuelve el día del año (1–365/366)."""
    return dt.timetuple().tm_yday


def equation_of_time(day):
    """
    Ecuación del tiempo (minutos).
    Formulación precisa.
    """
    gamma = 2*np.pi*(day - 1)/365
    Et = 229.18 * (
        0.000075
        + 0.001868*np.cos(gamma)
        - 0.032077*np.sin(gamma)
        - 0.014615*np.cos(2*gamma)
        - 0.040849*np.sin(2*gamma)
    )
    return Et  # minutos


def is_dst_madrid(dt):
    """
    Determina si aplica horario de verano en España (Europa).
    Regla: último domingo de marzo a último domingo de octubre.
    """
    year = dt.year

    # Último domingo de marzo
    march_last = max(week[-1] for week in __import__("calendar").monthcalendar(year, 3))
    dst_start = datetime(year, 3, march_last, 2)

    # Último domingo de octubre
    oct_last = max(week[-1] for week in __import__("calendar").monthcalendar(year, 10))
    dst_end = datetime(year, 10, oct_last, 3)

    return dst_start <= dt < dst_end


def solar_time_madrid(dt):
    """
    Convierte datetime (hora oficial Madrid) a hora solar verdadera.
    Devuelve:
        - datetime solar
        - Et (min)
        - día del año
    """
    n = day_of_year(dt)
    Et = equation_of_time(n)

    # Huso horario
    Z = 2 if is_dst_madrid(dt) else 1

    # Corrección total (minutos)
    correction = Et + 4*LON_MADRID - 60*Z

    dt_solar = dt + timedelta(minutes=correction)

    return dt_solar, Et, n




In [2]:
# Ejemplo de uso
dt = datetime(2026, 7, 1, 14, 0)  # 14:00 hora oficial
dt_solar, Et, n = solar_time_madrid(dt)

print(f"Día del año: {n}")
print(f"Ecuación del tiempo: {Et:.2f} min")
print(f"Hora solar: {dt_solar}")

Día del año: 182
Ecuación del tiempo: -3.46 min
Hora solar: 2026-07-01 11:41:43.348351
